# 🥇 Etapa 3: Gold - Modelo Dimensional (Star Schema)
## 🇧🇷 Português
### 🎯 O que este script faz?
Esta é a etapa final. Aqui, organizamos os dados no formato de **Fatos e Dimensões**. Este modelo é ideal para ferramentas de visualização (como Power BI) pois torna as buscas muito mais rápidas.

### 🏗️ Regras de Negócio e Estrutura:
1.  **Dim_Estudante:** Contém informações demográficas dos usuários (Idade, Gênero, País).
2.  **Dim_Plataforma:** Uma lista única de redes sociais com um código de identificação (ID).
3.  **Fact_Usage:** A tabela principal com os números. Ela conecta o estudante à plataforma e traz as métricas de horas de uso, sono e saúde mental.

---

# 🥇 Step 3: Gold - Dimensional Model (Star Schema)
## 🇺🇸 English (B1 Level)
### 🎯 What does this script do?
This is the final step. Here, we organize the data into **Facts and Dimensions**. This model is perfect for dashboard tools (like Power BI) because it makes queries much faster.

### 🏗️ Business Rules and Structure:
1.  **Dim_Estudante:** Contains user information (Age, Gender, Country).
2.  **Dim_Plataforma:** A unique list of social media platforms with an ID code.
3.  **Fact_Usage:** The main table with the metrics. It connects the student to the platform and shows hours of usage, sleep, and mental health scores.

---

### 📊 Por que usar esse modelo? / Why use this model?
* **Performance:** Filtros por país ou gênero ficam mais leves.
* **Organização:** Separa claramente o que é "descrição" (Dimensão) do que é "métrica" (Fato).

In [0]:
%sql
CREATE OR REPLACE TABLE gold.social_media.d_estudante AS
SELECT DISTINCT
    id_estudante,
    idade,
    faixa_etaria,
    genero,
    pais
FROM 
    silver.social_media.social_media_impact;

In [0]:
%sql
CREATE OR REPLACE TABLE gold.social_media.d_plataforma AS
SELECT DISTINCT
    dense_rank() OVER (ORDER BY plataforma_principal) AS id_plataforma,
    plataforma_principal AS nome_plataforma
FROM 
    silver.social_media.social_media_impact;

In [0]:
%sql
CREATE OR REPLACE TABLE gold.social_media.f_social_media_usage AS
SELECT
    s.id_estudante,
    p.id_plataforma,
    s.horas_uso_diario,
    s.is_heavy_user,
    s.impacto_academico_binario,
    s.horas_sono,
    -- Regra 1: Qualificação do Sono
    CASE 
        WHEN s.horas_sono < 6 THEN 'Privado de Sono'
        WHEN s.horas_sono BETWEEN 6 AND 8 THEN 'Sono Saudável'
        ELSE 'Sono Excelente'
    END AS classificacao_sono,
    s.score_saude_mental,
    -- Regra 2: Status de Saúde Mental
    CASE 
        WHEN s.score_saude_mental < 4 THEN 'Crítico'
        WHEN s.score_saude_mental BETWEEN 4 AND 7 THEN 'Alerta'
        ELSE 'Estável'
    END AS status_saude_mental,
    -- Regra 3: Eficiência do Dia (Horas Sono + Horas Uso)
    (s.horas_sono + s.horas_uso_diario) AS horas_dedicadas_telas_sono,
    s.data_processamento
FROM 
    silver.social_media.social_media_impact s
JOIN 
    gold.social_media.d_plataforma p ON s.plataforma_principal = p.nome_plataforma;